# 05 - Three strong baselines, three seeds each

Trains the reference models that SymptoGraph-LLM has to beat. All of them are
deliberately **flat**: they predict the targets directly, with no concept
bottleneck (N1), no comorbidity graph (N2) and no weak supervision (N3). The gap
between these and the proposed model is the contribution.

| Baseline | Encoder | Why this one |
|---|---|---|
| `tfidf_mlp` | TF-IDF (1-2 grams, fit on train only) into an MLP | classical bag-of-words reference; still competitive on short clinical text |
| `bilstm_attn` | trainable embeddings, BiLSTM, additive attention | the standard deep baseline in this literature, with no pretraining |
| `mentalroberta` | MentalRoBERTa fine-tuned | domain-pretrained transformer, the strongest published family on these corpora |

## Shared training protocol

Every baseline uses the **same multi-task head module, loss, masking, optimiser,
schedule, epoch budget, batch size, seeds, metrics and model-selection rule**.
Only the text encoder differs, so any difference in score is attributable to the
encoder rather than the training setup.

**One documented exception: the learning rate.** It is set per architecture
(1e-3 for the MLP and BiLSTM, 2e-5 for the transformer). A single shared value
would either diverge the transformer or barely move the MLP, and a crippled
baseline is worse than no baseline at all. Set `SHARED_LR` below to force one
value if you would rather have strict uniformity.

## Tasks and masking

Four heads are trained jointly: 22-way symptom multi-label, 4-class depression
severity, 5-class suicide risk, and binary stress. Targets are masked per row, so
a post with only severity labels contributes only to the severity loss. Symptom
targets use `1` present, `0` absent and `-1` not annotated, and `-1` never counts
as a negative.

## What gets written

```
baselines/
  tfidf_mlp/ | bilstm_attn/ | mentalroberta/
    seed_0/ seed_1/ seed_2/
      history.csv                epoch-wise, every metric, appended on resume
      checkpoint_last.pt         model + optimiser + scheduler + RNG state
      checkpoint_best.pt         best epoch by validation composite
      performance_metrics.json   final train/val/test bundle
      config.json
      plots/                     group curves, per-symptom curves, heatmap, bars
    aggregate/
      aggregate_metrics.json     mean and std across seeds
      summary.csv, summary_test.csv
      history_all_seeds.csv
      plots/
  comparison/
    leaderboard.md, leaderboard.csv, all_models.csv, comparison_headline.png
```

**Resumable.** A checkpoint is written at the end of every epoch. Re-running a
cell after a disconnect continues from the next epoch and **appends** to
`history.csv`; it never truncates or rewrites it.

Budget roughly **80 minutes on a T4** for all nine runs. Set `QUICK_TEST = True`
first to validate the whole pipeline in about two minutes.

## 1. Mount Drive and configure

In [ ]:
import os, sys, json, warnings
import pandas as pd
from google.colab import drive
warnings.filterwarnings('ignore')

drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/hari/SymptoGraph-LLM'
DRIVE_PROCESSED = f'{DRIVE_BASE}/processed'
DRIVE_RESULTS = f'{DRIVE_BASE}/baselines'
LOCAL_PROCESSED, LOCAL_RESULTS = '/content/processed', '/content/baselines'

# True -> tiny subset, 2 epochs, tiny transformer. Validates the pipeline fast.
QUICK_TEST = True
# Force one learning rate for every baseline. None keeps the per-architecture rates.
SHARED_LR = None

os.environ.update(DRIVE_PROCESSED=DRIVE_PROCESSED, DRIVE_RESULTS=DRIVE_RESULTS,
                  LOCAL_PROCESSED=LOCAL_PROCESSED, LOCAL_RESULTS=LOCAL_RESULTS)
assert os.path.isdir(DRIVE_PROCESSED), f'{DRIVE_PROCESSED} not found - run notebook 02'
!mkdir -p $LOCAL_PROCESSED $LOCAL_RESULTS
!rsync -ah "$DRIVE_PROCESSED/" "$LOCAL_PROCESSED/"
# pull back any partial results so a previous session can be resumed
!test -d "$DRIVE_RESULTS" && rsync -ah "$DRIVE_RESULTS/" "$LOCAL_RESULTS/" || true
print('processed:', sorted(os.listdir(LOCAL_PROCESSED)))
print('existing results:', sorted(os.listdir(LOCAL_RESULTS)))

## 2. Get the code and check the GPU

In [ ]:
REPO = 'https://github.com/sid-2k6/SymptoGraph-LLM.git'
CHECKOUT = '/content/SymptoGraph-LLM'

if os.path.isdir(f'{CHECKOUT}/.git'):
    !git -C $CHECKOUT fetch -q origin && git -C $CHECKOUT reset -q --hard origin/main
else:
    !git clone -q $REPO $CHECKOUT

need = ['src/symptograph/baselines.py', 'src/symptograph/metrics.py']
missing = [f for f in need if not os.path.isfile(f'{CHECKOUT}/{f}')]
assert not missing, f'stale checkout, missing {missing} - run: !rm -rf {CHECKOUT}'
sys.path.insert(0, f'{CHECKOUT}/src')
!git -C $CHECKOUT log --oneline -1

In [ ]:
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu  ', torch.cuda.get_device_name(0))
else:
    print('No GPU. Runtime -> Change runtime type -> T4 GPU, or expect a long run.')

## 3. The shared training configuration

One object drives all nine runs, so the protocol is verifiable rather than
described. It is saved into every run folder as `config.json`.

In [ ]:
from symptograph.baselines import TrainConfig, BASELINES, run_all, run_baseline
from symptograph.metrics import build_leaderboard, headline_table
from symptograph.preprocess import Ontology

onto = Ontology.load(f'{CHECKOUT}/configs/symptom_ontology.json')

if QUICK_TEST:
    cfg = TrainConfig(epochs=2, seeds=(0, 1, 2), max_length=64,
                      limit_train=500, limit_eval=300,
                      tfidf_features=2000, vocab_size=2000,
                      embed_dim=32, hidden_dim=32, amp=False,
                      transformer_name='prajjwal1/bert-tiny',
                      shared_lr=SHARED_LR)
    RESULTS = f'{LOCAL_RESULTS}_quicktest'
else:
    cfg = TrainConfig(epochs=8, batch_size=16, eval_batch_size=64,
                      max_length=256, seeds=(0, 1, 2),
                      shared_lr=SHARED_LR)
    RESULTS = LOCAL_RESULTS

print('output root:', RESULTS)
print('baselines  :', BASELINES)
print('seeds      :', cfg.seeds)
print('epochs     :', cfg.epochs, '| batch', cfg.batch_size,
      '| max_length', cfg.max_length)
print('learning rates:', {b: cfg.lr_for(b) for b in BASELINES})
print('task weights  :', cfg.task_weights)
print('selection     :', cfg.selection_metric, '(validation)')

### What the data looks like

Note how few rows carry severity and especially risk labels. Risk has only 340
training and 74 test rows, so its QWK will be noisy across seeds - which is
exactly why three seeds and a reported standard deviation matter.

In [ ]:
lab = pd.read_parquet(f'{LOCAL_PROCESSED}/labelled.parquet')
sym_cols = onto.columns
rows = []
for split in ('train', 'val', 'test'):
    s = lab[lab.split == split]
    rows.append({'split': split, 'posts': len(s),
                 'symptom_rows': int((s[sym_cols] >= 0).any(axis=1).sum()),
                 'severity': int(s.severity_depression.notna().sum()),
                 'risk': int(s.severity_risk.notna().sum()),
                 'stress': int(s.stress.notna().sum())})
display(pd.DataFrame(rows).set_index('split'))
print('symptoms with gold labels:',
      int(((lab[sym_cols] >= 0).sum() > 0).sum()), 'of', len(sym_cols))

## 4. Train

Nine runs: three baselines times three seeds. Progress prints per epoch.

**If the session drops, just re-run this cell.** Finished epochs are skipped,
and `history.csv` is appended to rather than rewritten. Run section 7 afterwards
even on a partial pass, so checkpoints reach Drive.

In [ ]:
results = run_all(cfg, LOCAL_PROCESSED, RESULTS,
                  baselines=BASELINES, onto=onto, resume=True, verbose=True)
print('\ndone:', list(results))

## 5. Per-baseline results

`history.csv` holds every metric for every epoch, for train and validation.
Around 300 columns: per-symptom F1, precision, recall and AUROC, the group
aggregates, and each task's metrics.

In [ ]:
for name in BASELINES:
    h = pd.read_csv(f'{RESULTS}/{name}/seed_0/history.csv')
    print(f'\n=== {name} (seed 0) - {len(h)} epochs, {len(h.columns)} logged columns ===')
    cols = ['epoch', 'train_loss', 'val_loss', 'train_group_all_macro_f1',
            'val_group_all_macro_f1', 'val_severity_depression_qwk',
            'val_severity_risk_qwk', 'val_composite']
    display(h[[c for c in cols if c in h.columns]].round(4))

In [ ]:
from IPython.display import Image, display as show

for name in BASELINES:
    print(f'\n{"=" * 70}\n{name} - seed 0\n{"=" * 70}')
    for plot in ('curves_group.png', 'bars_per_symptom.png',
                 'heatmap_per_symptom_val.png'):
        p = f'{RESULTS}/{name}/seed_0/plots/{plot}'
        if os.path.exists(p):
            show(Image(filename=p))

## 6. Mean and standard deviation across seeds

The numbers that go in the paper. `weak_only` symptoms show `-` because these
baselines train on gold labels only, so those eleven symptoms have no supervision
at all - that absence is the gap novelty N3 is meant to fill.

In [ ]:
for name in BASELINES:
    agg = json.load(open(f'{RESULTS}/{name}/aggregate/aggregate_metrics.json'))
    print(f'\n=== {name} === encoder: {agg["encoder"]}'
          f' | best epoch per seed: {agg["per_seed_best_epoch"]}')
    display(headline_table(agg['aggregate'], label=name)[['metric', 'formatted']]
            .set_index('metric'))

In [ ]:
for name in BASELINES:
    print(f'\n{name} - aggregate plots')
    for plot in ('aggregate_group_metrics.png', 'aggregate_per_symptom.png'):
        p = f'{RESULTS}/{name}/aggregate/plots/{plot}'
        if os.path.exists(p):
            show(Image(filename=p))

## 7. Leaderboard

`build_leaderboard` scans every subfolder containing
`aggregate/aggregate_metrics.json`. When the proposed model writes its results in
the same place and format, it appears here automatically with no code changes.

In [ ]:
combined, tables = build_leaderboard(RESULTS, f'{RESULTS}/comparison')
wide = combined.pivot(index='metric', columns='model', values='formatted')
display(wide)

print('\nvalidation composite, ranked:')
rank = (combined[combined.metric == 'composite']
        .sort_values('mean', ascending=False)[['model', 'formatted']])
display(rank.set_index('model'))
show(Image(filename=f'{RESULTS}/comparison/comparison_headline.png'))

### Held-out test results

Reported from the **best-validation checkpoint** of each seed, so the test split
is never used for model selection.

In [ ]:
test_rows = []
for name in BASELINES:
    agg = json.load(open(f'{RESULTS}/{name}/aggregate/aggregate_metrics.json'))
    t = headline_table(agg['aggregate_by_split']['test'], label=name)
    test_rows.append(t)
test_combined = pd.concat(test_rows, ignore_index=True)
display(test_combined.pivot(index='metric', columns='model', values='formatted'))
test_combined.to_csv(f'{RESULTS}/comparison/all_models_test.csv', index=False)

## 8. Save everything to Drive

Safe on a partial run. Copy the checkpoints too, since they are what makes the
next session resumable.

In [ ]:
!mkdir -p "$DRIVE_RESULTS"
SRC = RESULTS
!rsync -ah --info=progress2 "$SRC/" "$DRIVE_RESULTS/"
!du -sh "$DRIVE_RESULTS"/* | sort -h
print()
print(open(f'{RESULTS}/comparison/leaderboard.md').read()[:1500])

## Notes

**Model selection.** `composite` is the mean of symptom macro-F1, depression
severity QWK and suicide risk QWK on validation. One number is needed to pick a
checkpoint; all three parts are logged separately so you can report whichever the
paper needs.

**Train metrics** are accumulated from the same forward passes used for the
gradient step, so they cost nothing extra. They are therefore measured with
dropout active and are slightly noisier than a clean pass - fine for spotting
overfitting, which is what the train-versus-val curves are for.

**Risk metrics are noisy.** With 340 training and 74 test rows, suicide risk QWK
varies substantially across seeds. Always quote it with its standard deviation,
and avoid drawing conclusions from differences smaller than it.

### Plugging the proposed model into this comparison

Write the proposed model's results to `<root>/symptograph_llm/` with the same
layout, then re-run section 7. To stay comparable, it must use the same
`TrainConfig`, the same seeds, the same splits, and the same `compute_metrics`
call:

```python
from symptograph.metrics import compute_metrics, aggregate_seeds, headline_table
bundle = compute_metrics(preds, targets, onto)   # identical metric definitions
```

**For a fair N3 comparison**, also run these baselines with
`use_weak_labels=True` and `weak_label_path` pointing at notebook 04's output,
writing to a separate root such as `baselines_weak/`. Otherwise the proposed
model gains from weak supervision that the baselines never saw, and the
comparison overstates the architectural contribution:

```python
cfg_weak = TrainConfig(
    **{**cfg.__dict__, 'use_weak_labels': True,
       'weak_label_path': f'{DRIVE_BASE}/weak_labels/weak_labels.parquet'})
run_all(cfg_weak, LOCAL_PROCESSED, f'{LOCAL_RESULTS}_weak', onto=onto)
```

**Next:** notebook 06 trains SymptoGraph-LLM itself - the concept bottleneck (N1)
over the graph from notebook 03 (N2), with the confidence-weighted weak labels
from notebook 04 (N3).